# RAG 파이프라인 (시나리오 B: LLM API 기반)

공공 IT 입찰 RFP 문서를 대상으로 한 RAG 시스템 베이스라인 구축

**기술 스택**
- LLM: OpenAI GPT-4o-mini
- 임베딩: OpenAI text-embedding-3-small
- VectorDB: Chroma
- 청킹: LangChain RecursiveCharacterTextSplitter

**파이프라인 순서**
1. 전처리 → 2. 청킹 → 3. 임베딩 + VectorDB → 4. 검색 + 답변 생성

In [ ]:
import os
import re
import pandas as pd
import chromadb
from dotenv import load_dotenv
from openai import OpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

# OpenAI API 키 로드
load_dotenv(os.path.join(os.getcwd(), '.env'))
client = OpenAI()

EMBEDDING_MODEL = 'text-embedding-3-small'
LLM_MODEL = 'gpt-4o-mini'
DB_PATH = './vectordb'

print("환경 설정 완료")

## 1단계: 전처리

EDA에서 발견한 노이즈를 제거합니다.
- 규칙 1: 연속공백(3개+) → 줄바꿈 변환
- 규칙 2: 5자 이하 단독줄 제거 (머리글/꼬리글)
- 규칙 3: 페이지번호 패턴 제거 (`- 1 -` 등)

In [ ]:
def clean_text(text: str) -> str:
    """EDA 인사이트 기반 텍스트 전처리"""
    # 규칙 1: 연속공백(3개+) → 단일 줄바꿈
    text = re.sub(r' {3,}', '\n', text)
    # 규칙 3: 페이지번호 패턴 제거
    text = re.sub(r'^\s*-?\s*\d{1,3}\s*-?\s*$', '', text, flags=re.MULTILINE)
    # 규칙 2: 5자 이하 단독줄 제거
    lines = text.split('\n')
    lines = [line for line in lines if len(line.strip()) > 5 or len(line.strip()) == 0]
    # 연속 빈줄 정리
    text = '\n'.join(lines)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

# 원본 데이터 로드 및 전처리
df = pd.read_csv('data_list.csv', encoding='utf-8-sig')
df['텍스트_원본'] = df['텍스트']
df['텍스트'] = df['텍스트'].apply(clean_text)

# 전처리 결과
df['원본_길이'] = df['텍스트_원본'].str.len()
df['전처리_길이'] = df['텍스트'].str.len()
df['축소율'] = ((df['원본_길이'] - df['전처리_길이']) / df['원본_길이'] * 100).round(1)

print(f"원본 데이터: {len(df)}건")
print(f"평균 축소율: {df['축소율'].mean():.1f}%")
print(f"평균 길이: {df['원본_길이'].mean():,.0f}자 → {df['전처리_길이'].mean():,.0f}자")

## 2단계: 청킹

전처리된 텍스트를 1,000자 단위로 분할하고, 각 청크에 메타데이터를 부착합니다.
- chunk_size: 1,000자
- chunk_overlap: 200자 (문맥 유지)
- 500자 미만 문서는 분할하지 않고 통째로 유지
- 메타데이터: 사업명, 발주기관, 금액구간, 공개연도

In [ ]:
def classify_amount(amount):
    """사업 금액을 구간으로 분류"""
    if pd.isna(amount) or amount == 0:
        return "미정"
    elif amount < 1e8:
        return "소규모(<1억)"
    elif amount < 5e8:
        return "중규모(1~5억)"
    else:
        return "대규모(>5억)"

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=['\n\n', '\n', ' '],
)

chunks = []
for _, row in df.iterrows():
    text = str(row['텍스트'])
    # 500자 미만은 통째로 유지
    if len(text) < 500:
        text_chunks = [text]
    else:
        text_chunks = splitter.split_text(text)

    pub_year = str(row['공개 일자'])[:4] if pd.notna(row['공개 일자']) else "미상"
    for i, chunk in enumerate(text_chunks):
        chunks.append({
            'text': chunk,
            'metadata': {
                '사업명': str(row['사업명']),
                '발주기관': str(row['발주 기관']),
                '금액구간': classify_amount(row.get('사업 금액')),
                '공개연도': pub_year,
                '청크번호': f"{i+1}/{len(text_chunks)}",
            }
        })

lengths = [len(c['text']) for c in chunks]
print(f"청크 수: {len(chunks)}개")
print(f"평균 청크 길이: {sum(lengths)/len(lengths):,.0f}자")
print(f"최소/최대: {min(lengths)}자 / {max(lengths)}자")

## 3단계: 임베딩 + VectorDB 구축

OpenAI `text-embedding-3-small`로 각 청크를 1,536차원 벡터로 변환한 뒤, Chroma DB에 저장합니다.

In [ ]:
def get_embeddings(texts, batch_size=50):
    """OpenAI API로 텍스트 리스트를 임베딩"""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
        print(f"  임베딩 완료: {min(i + batch_size, len(texts))}/{len(texts)}")
    return all_embeddings

# Chroma DB 생성
chroma_client = chromadb.PersistentClient(path=DB_PATH)
try:
    chroma_client.delete_collection('rfp_documents')
except Exception:
    pass
collection = chroma_client.create_collection(
    name='rfp_documents',
    metadata={'hnsw:space': 'cosine'}
)

# 임베딩 생성 및 저장
texts = [c['text'] for c in chunks]
print(f"임베딩 생성 중... (총 {len(texts)}개 청크)")
embeddings = get_embeddings(texts)

collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    embeddings=embeddings,
    documents=texts,
    metadatas=[c['metadata'] for c in chunks],
)

print(f"VectorDB 저장 완료: {collection.count()}개 청크 → {DB_PATH}")

## 4단계: 검색 + 답변 생성

질문이 들어오면 하이브리드 검색(키워드 + 의미)으로 관련 청크를 찾고, GPT-4o-mini가 답변을 생성합니다.

**하이브리드 검색이란?**
- 키워드 검색: "한영대학교"처럼 고유명사가 있으면 해당 단어가 포함된 문서를 바로 찾음
- 의미 검색: 질문을 벡터로 변환해서 가장 비슷한 문서를 찾음
- 두 결과를 합쳐서 정확도를 높임

In [ ]:
SYSTEM_PROMPT = """당신은 공공 IT 입찰 RFP(제안요청서) 전문 컨설턴트입니다.
제공된 문서 내용을 바탕으로 정확하고 구체적으로 답변하세요.

규칙:
1. 반드시 제공된 문맥(Context)에 있는 내용만 기반으로 답변하세요.
2. 문맥에 없는 내용은 "제공된 문서에서 해당 정보를 찾을 수 없습니다."라고 답하세요.
3. 답변 시 출처가 되는 사업명을 함께 언급하세요.
4. 금액, 일정, 기술 요구사항 등 구체적인 수치가 있다면 정확히 인용하세요."""


def extract_keywords(query):
    """질문에서 고유명사 키워드 추출"""
    patterns = [
        r'[\w]+대학교?',
        r'[\w]+광역시',
        r'[\w]+특별시',
        r'[\w]+[시군구](?=\s|$)',
    ]
    keywords = []
    for p in patterns:
        keywords.extend(re.findall(p, query))
    return keywords


def search(query, k=5):
    """하이브리드 검색: 키워드 필터 + 의미 검색"""
    chroma_client = chromadb.PersistentClient(path=DB_PATH)
    collection = chroma_client.get_collection('rfp_documents')

    # 1) 키워드 검색
    keywords = extract_keywords(query)
    keyword_docs = []
    for kw in keywords:
        try:
            results = collection.get(
                where_document={'$contains': kw},
                include=['metadatas', 'documents'],
            )
            for i in range(len(results['ids'])):
                keyword_docs.append({
                    'id': results['ids'][i],
                    'text': results['documents'][i],
                    'metadata': results['metadatas'][i],
                    'similarity': 1.0,
                    'match_type': '키워드',
                })
        except Exception:
            pass

    # 2) 의미 검색
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=query)
    query_embedding = response.data[0].embedding
    sem_results = collection.query(query_embeddings=[query_embedding], n_results=k)

    semantic_docs = []
    for i in range(len(sem_results['documents'][0])):
        semantic_docs.append({
            'id': sem_results['ids'][0][i],
            'text': sem_results['documents'][0][i],
            'metadata': sem_results['metadatas'][0][i],
            'similarity': 1 - sem_results['distances'][0][i],
            'match_type': '의미',
        })

    # 3) 결과 합치기 (키워드 우선, 중복 제거)
    seen_ids = set()
    merged = []
    for doc in keyword_docs + semantic_docs:
        if doc['id'] not in seen_ids:
            seen_ids.add(doc['id'])
            merged.append(doc)
    return merged[:k]


def generate(query, docs):
    """검색된 문서를 컨텍스트로 LLM 답변 생성"""
    if not docs:
        return "제공된 문서에서 해당 정보를 찾을 수 없습니다."

    context_parts = []
    for i, doc in enumerate(docs, 1):
        meta = doc['metadata']
        context_parts.append(
            f"[문서 {i}] 사업명: {meta['사업명']} | 발주기관: {meta['발주기관']} | 금액구간: {meta['금액구간']}\n"
            f"{doc['text']}"
        )
    context = "\n\n".join(context_parts)

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"[Context]\n{context}\n\n[질문]\n{query}"},
        ],
        temperature=0,
    )
    return response.choices[0].message.content


def ask(query, k=5):
    """RAG 전체 파이프라인: 질문 → 검색 → 답변"""
    print(f"질문: {query}")
    docs = search(query, k=k)
    print(f"검색 결과: {len(docs)}개 청크")
    for i, doc in enumerate(docs, 1):
        print(f"  [{i}] ({doc['match_type']}) 유사도: {doc['similarity']:.4f} | {doc['metadata']['사업명']}")
    answer = generate(query, docs)
    print(f"\n답변:\n{answer}\n")
    return answer

print("검색 + 답변 생성 함수 정의 완료")

## 테스트

In [ ]:
ask("한영대학교 사업의 사업기간과 예산은?")

In [ ]:
ask("버스정보시스템 구축 사업에서 요구하는 주요 기능은?")

In [ ]:
ask("울산광역시가 발주한 사업의 입찰 방법은?")